In [1]:
# !pip install vega_datasets

In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
import datetime as dt
from calitp_data_analysis.sql import get_engine
# from shared_utils import gtfs_utils_v2
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()
from shapely import wkt

pd.set_option('display.max_columns', None)

In [3]:
# GCS FILE PATH
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
# Load the stored ACS dataset from the specified GCS file path.
with fs.open(f"{GCS_FILE_PATH}/census_blocks_data.parquet", "rb") as f:
    blocks_ca_acs = gpd.read_parquet(f)

# Load job density data from GCS and select required columns
# Open the GCS file using your existing fs object
with fs.open(f"{GCS_FILE_PATH}/job_density_blockwithrac_2023.parquet", "rb") as f:
    jobdata = pd.read_parquet(f)

# Select only the columns you want, including geometry
jobdata = jobdata[['GEOID', 'jobs_tot_work', 'jobs_tot_home' ]]

# Load pois data from GCS and select required columns
with fs.open(f"{GCS_FILE_PATH}/pois_2026.parquet", "rb") as f:
    pois = gpd.read_parquet(f)

In [5]:
# agencies_to_find = [
#     "Marin Optibus Schedule",
# ]

# agency_sql = ', '.join(f"'{agency}'" for agency in agencies_to_find)

In [6]:
agencies_to_find = [
    "Lake Schedule",
    "Humboldt Schedule",
    "Trinity Schedule",
    "Lassen Schedule",
    "B-Line Schedule",
    "TART, North Lake Tahoe Schedule",
    "El Dorado Schedule",
    "Bay Area 511 Santa Clara Transit Schedule",
    "Bay Area 511 Marin Schedule",
    "Bay Area 511 Vine Transit Schedule",
    "Monterey Salinas Schedule",
    "Santa Maria Schedule",
    "Kern Schedule",
    "TCRTA Schedule",
    "Antelope Valley Transit Authority Schedule",
    "VCTC Schedule",
    "LA Metro Bus Schedule",
    "LA Metro Rail Schedule",
    "Mountain Transit Schedule",
    "Morongo Basin Schedule",
    "Victor Valley Schedule",
    "Eastern Sierra Schedule",
    "StanRTA Schedule",
    "YARTS Schedule",
    "North County Schedule",
    "Imperial Valley Transit Schedule",
]

agency_sql = ', '.join(f"'{agency}'" for agency in agencies_to_find)

In [7]:
with db_engine.connect() as connection:
    query = f"""
        SELECT *
        FROM `cal-itp-data-infra.mart_gtfs.fct_daily_schedule_feeds`
        WHERE gtfs_dataset_name IN ({agency_sql})
          AND date = DATE('2026-05-21')
    """
    schedule_feeds = pd.read_sql(query, connection)

/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/google/cloud/bigquery/client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


In [8]:
schedule_feeds.head(3)

,key,date,feed_key,feed_timezone,base64_url,gtfs_dataset_key,gtfs_dataset_name
0,e5ea4687844e002620a781843d4de9e0,2026-05-21,b94f8d842a2be5121b09f94762e5b2b3,America/Los_Angeles,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,0089bd1b0a2b78a8590d8749737d7146,Bay Area 511 Marin Schedule
1,1ac89eb76064a005633ca4907c58380b,2026-05-21,4544f539dd546b5227cbefca004a5f95,America/Los_Angeles,aHR0cHM6Ly93d3cubXN0Lm9yZy9nb29nbGUvZ29vZ2xlX3...,076e30b080fdc5501151bd3fb0a37b9e,Monterey Salinas Schedule
2,4e29e2dda8a591710994e6fd681a122c,2026-05-21,2dfab4dd09aa756be1e7139cd5b411f2,America/Los_Angeles,aHR0cHM6Ly9kYXRhLnRyaWxsaXVtdHJhbnNpdC5jb20vZ3...,0961fd01babae0388e6e110264d2f434,Morongo Basin Schedule


In [9]:
# Get all unique feed_keys across the 3 agencies
feed_keys = schedule_feeds["feed_key"].unique().tolist()
feed_keys_str = ", ".join(f"'{k}'" for k in feed_keys)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, daily_arrivals, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2026-05-21')
          AND feed_key IN ({feed_keys_str})
    """
    stops_unique_weekday = pd.read_sql(query, connection)

# Map feed_key -> gtfs_dataset_name so we can split by agency
feed_key_to_agency = schedule_feeds.set_index("feed_key")["gtfs_dataset_name"].to_dict()
stops_unique_weekday["gtfs_dataset_name"] = stops_unique_weekday["feed_key"].map(feed_key_to_agency)


In [10]:
stops_unique_weekday.head(2)

,feed_key,stop_id,_feed_valid_from,n_hours_in_service,daily_arrivals,arrivals_early_am,arrivals_am_peak,arrivals_midday,arrivals_pm_peak,arrivals_evening,route_id_array,route_type_array,stop_key,tts_stop_name,pt_geom,stop_name,location_type,stop_desc,stop_code,gtfs_dataset_name
0,56f5af24bdee205a12acfc29e8d2406a,895,2026-02-13 03:07:54.583880+00:00,1,1,0,3,0,0,0,[94],[3],0c2243d0f5b5b4019db07823f03f79b0,None,POINT(-118.1479873 34.69064602),10th St. W. & Ave. J,0.0,None,895,Antelope Valley Transit Authority Schedule
1,56f5af24bdee205a12acfc29e8d2406a,797,2026-02-13 03:07:54.583880+00:00,1,1,0,3,0,0,0,[97],[3],f4dca0b8c89bfc7ebc311a715307875d,None,POINT(-118.147469 34.598623),10th St. W. & Marketplace Dr.,0.0,None,797,Antelope Valley Transit Authority Schedule


In [11]:
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, route_id, route_short_name, route_long_name, route_type
        FROM cal-itp-data-infra.mart_gtfs.dim_routes
        WHERE feed_key IN ({feed_keys_str})
    """
    route_lookup = pd.read_sql(query, connection)

# Map feed_key -> agency name so it's clear which routes belong to which agency
route_lookup["organization_name"] = route_lookup["feed_key"].map(feed_key_to_agency)

In [12]:
# Keep route_id as string
route_lookup['route_id'] = route_lookup['route_id'].astype(str)

short_name_map = dict(zip(route_lookup['route_id'], route_lookup['route_short_name']))
long_name_map  = dict(zip(route_lookup['route_id'], route_lookup['route_long_name']))

def map_route_names(route_id_array, name_map):
    if not isinstance(route_id_array, list) or len(route_id_array) == 0:
        return None
    return [name_map.get(str(rid), None) for rid in route_id_array]

stops_unique_weekday['route_short_name_list'] = stops_unique_weekday['route_id_array'].apply(
    lambda x: map_route_names(x, short_name_map)
)

stops_unique_weekday['route_long_name_list'] = stops_unique_weekday['route_id_array'].apply(
    lambda x: map_route_names(x, long_name_map)
)

In [13]:
def expand_routes_fixed(row):
    """
    Expands a stop row with multiple route_ids and route_types into separate rows
    for each route, keeping existing pt_geom and stop_code.

    Uses existing daily_arrivals as arrivals_all_day.
    """

    # Ensure arrays exist
    route_ids = row['route_id_array'] if isinstance(row['route_id_array'], list) else []
    route_types = row['route_type_array'] if isinstance(row['route_type_array'], list) else []

    # If only one route_type but multiple route_ids, replicate route_type
    if len(route_types) == 1 and len(route_ids) > 1:
        route_types = route_types * len(route_ids)

    # Pair routes with types
    pairs = list(zip(route_ids, route_types))

    # Pull values from row
    stop_code_value = row.get('stop_code', pd.NA)
    pt_geom_value = row.get('pt_geom', pd.NA)
    daily_arrivals = row.get('daily_arrivals', 0)

    # Handle case with no routes
    if not pairs:
        return pd.DataFrame({
            'gtfs_dataset_name': [row['gtfs_dataset_name']],
            'feed_key': [row['feed_key']],
            'stop_id': [row['stop_id']],
            'stop_code': [stop_code_value],
            'stop_name': [row['stop_name']],
            'route_id': [pd.NA],
            'route_type': [pd.NA],
            'pt_geom': [pt_geom_value],
            'arrivals_all_day': [daily_arrivals]
        })

    # Expand into multiple rows
    return pd.DataFrame({
        'gtfs_dataset_name': [row['gtfs_dataset_name']] * len(pairs),
        'feed_key': [row['feed_key']] * len(pairs),
        'stop_id': [row['stop_id']] * len(pairs),
        'stop_code': [stop_code_value] * len(pairs),
        'stop_name': [row['stop_name']] * len(pairs),
        'route_id': [r[0] for r in pairs],
        'route_type': [r[1] for r in pairs],
        'pt_geom': [pt_geom_value] * len(pairs),
        'arrivals_all_day': [daily_arrivals] * len(pairs)
    })

In [14]:
stops_expanded_weekday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_weekday.iterrows()], ignore_index=True)

In [15]:
def aggregate_stops_listcodes(stops_expanded_df):
    stops_expanded_df['route_type'] = stops_expanded_df['route_type'].astype(str)
    stops_expanded_df['route_id'] = stops_expanded_df['route_id'].astype(str)
    # map route_type to mode bucket
    rail_types = {'0', '1', '2', '12'}  # tram/streetcar/light rail, subway, rail, monorail
    ferry_types = {'4'}
    bus_types = {'3', '11'}
    def to_mode(rt):
        if rt in rail_types:
            return 'rail'
        elif rt in ferry_types:
            return 'ferry'
        elif rt in bus_types:
            return 'bus'
        else:
            return 'other'
    stops_expanded_df['mode_group'] = stops_expanded_df['route_type'].apply(to_mode)
    # overall aggregation (unchanged - keep this as your IV instrument)
    stops_aggregated = stops_expanded_df.groupby(
        ['feed_key', 'stop_id', 'stop_name'], dropna=False
    ).agg(
        gtfs_dataset_name = ('gtfs_dataset_name', 'first'),
        n_arrivals=('arrivals_all_day', 'first'),
        n_routes=('route_id', 'nunique'),
        route_id_list=('route_id', lambda x: sorted(set(x))),
        stop_code=('stop_code', 'first'),
        pt_geom=('pt_geom', lambda x: next((v for v in x if pd.notna(v)), pd.NA))
    ).reset_index()
    # mode-specific route counts, pivoted wide
    mode_counts = (
        stops_expanded_df.groupby(['feed_key', 'stop_id', 'mode_group'])['route_id']
        .nunique()
        .unstack(fill_value=0)
        .add_prefix('n_routes_')
        .reset_index()
    )
    stops_aggregated = stops_aggregated.merge(
        mode_counts, on=['feed_key', 'stop_id'], how='left'
    )
    # fill any mode columns that didn't appear at all in this batch
    for col in ['n_routes_bus', 'n_routes_rail', 'n_routes_ferry', 'n_routes_other']:
        if col not in stops_aggregated.columns:
            stops_aggregated[col] = 0
        stops_aggregated[col] = stops_aggregated[col].fillna(0).astype(int)
    return stops_aggregated

In [16]:
agg_with_route_weekday = aggregate_stops_listcodes(stops_expanded_weekday)

In [17]:
agg_with_route_weekday.head(2)

,feed_key,stop_id,stop_name,gtfs_dataset_name,n_arrivals,n_routes,route_id_list,stop_code,pt_geom,n_routes_bus,n_routes_rail,n_routes_ferry,n_routes_other
0,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0
1,019262d9a8ecd3ff0b48b60c5d727ea7,1002,9th St & Nunes Rd,StanRTA Schedule,15,1,[290],1002,POINT(-120.905698 37.551516),1,0,0,0


In [18]:
# agg_with_route_weekday['name'] = agg_with_route_weekday['feed_key'].map(feed_key_to_schedule)

In [19]:
# Drop rows with missing pt_geom
agg_with_route_weekday = agg_with_route_weekday[
    agg_with_route_weekday['pt_geom'].notna() & 
    (agg_with_route_weekday['pt_geom'] != 'nan')
].copy()

# Ensure pt_geom is string type
agg_with_route_weekday['pt_geom'] = agg_with_route_weekday['pt_geom'].astype(str)

In [20]:
# Convert pt_geom column from WKT to shapely geometry
agg_with_route_weekday['geometry'] = agg_with_route_weekday['pt_geom'].apply(wkt.loads)

# # Set CRS (assuming WGS84)
# agg_with_route_weekday.set_crs(epsg=4326, inplace=True)

agg_with_route_weekday = gpd.GeoDataFrame(
    agg_with_route_weekday,
    geometry="geometry",
    crs="EPSG:4326"
)

In [21]:
# Reproject to match census tracts CRS
agg_with_route_weekday = agg_with_route_weekday.to_crs(blocks_ca_acs.crs)

In [22]:
stop_buffered = agg_with_route_weekday.copy()
stop_buffered["geometry"] = stop_buffered.geometry.buffer(404.672)

In [23]:
blocks_ca_acs['lowincome_total'] = (
    blocks_ca_acs['inc_extremelylow'] + 
    blocks_ca_acs['inc_verylow'] 
)

In [24]:
blocks_ca_acs.head(2)

,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOIDFQ,GEOID,NAME,NAMELSAD,LSAD,ALAND,AWATER,geometry,state,county,block group,county_name,total_pop,median_household_income,employed_pop,households_no_vehicle,total_youth,inc_extremelylow,inc_verylow,inc_low,inc_total_lowincome,area_m2,lowincome_total
0,06,073,010601,1,1500000US060730106011,060730106011,1,Block Group 1,BG,739477,3511,"POLYGON ((268816.912 -592917.261, 268817.085 -...",6,73,1,Census Tract 106.01,929,211875,457,0,87,29,14,9,52,7.823089e+05,43
1,06,079,013000,1,1500000US060790130001,060790130001,1,Block Group 1,BG,724518353,27447720,"MULTIPOLYGON (((-106426.951 -247177.326, -1065...",6,79,1,Census Tract 130,1760,82021,657,0,272,82,70,123,275,7.238044e+08,152


In [25]:
# Inner join with ACS data on 'geo_id'
blocks_ca_acs = blocks_ca_acs.merge(jobdata, on = 'GEOID', how='left')
pois = pois.to_crs(blocks_ca_acs.crs)
pois_with_block = gpd.sjoin(
    pois,
    blocks_ca_acs[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)


poi_total = (
    pois_with_block
    .groupby("GEOID")
    .size()
    .reset_index(name="poi_total")
)

blocks_ca_acs = blocks_ca_acs.merge(poi_total, on="GEOID", how="left")
blocks_ca_acs["poi_total"] = blocks_ca_acs["poi_total"].fillna(0)


In [26]:
geometry_intersect = gpd.overlay(
    stop_buffered, 
    blocks_ca_acs, 
    how='intersection', 
    keep_geom_type=True
)

In [27]:
geometry_intersect.head(5)

,feed_key,stop_id,stop_name,gtfs_dataset_name,n_arrivals,n_routes,route_id_list,stop_code,pt_geom,n_routes_bus,n_routes_rail,n_routes_ferry,n_routes_other,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOIDFQ,GEOID,NAME,NAMELSAD,LSAD,ALAND,AWATER,state,county,block group,county_name,total_pop,median_household_income,employed_pop,households_no_vehicle,total_youth,inc_extremelylow,inc_verylow,inc_low,inc_total_lowincome,area_m2,lowincome_total,jobs_tot_work,jobs_tot_home,poi_total,geometry
0,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,06,099,001604,2,1500000US060990016042,060990016042,2,Block Group 2,BG,216338,0,6,99,2,Census Tract 16.04,891,19453,242,61,370,221,65,22,308,216030.070602,286,142.0,476.0,0.0,"POLYGON ((-89193.486 -44269.998, -89233.150 -4..."
1,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,06,099,001601,3,1500000US060990016013,060990016013,3,Block Group 3,BG,379341,0,6,99,3,Census Tract 16.01,2126,55787,1007,8,709,53,147,135,335,369521.657418,200,110.0,541.0,5.0,"POLYGON ((-89384.247 -43508.437, -89348.347 -4..."
2,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,06,099,002200,2,1500000US060990022002,060990022002,2,Block Group 2,BG,613012,0,6,99,2,Census Tract 22,2252,59265,1117,65,617,154,43,225,422,603372.551139,197,143.0,801.0,1.0,"POLYGON ((-89153.821 -43462.602, -89114.538 -4..."
3,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,06,099,001604,1,1500000US060990016041,060990016041,1,Block Group 1,BG,265624,0,6,99,1,Census Tract 16.04,1911,41420,507,26,959,99,122,91,312,266310.591029,221,35.0,494.0,1.0,"POLYGON ((-89590.382 -43944.273, -89596.209 -4..."
4,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,06,099,002200,4,1500000US060990022004,060990022004,4,Block Group 4,BG,433184,0,6,99,4,Census Tract 22,1250,49712,626,25,462,92,74,56,222,426892.496973,166,39.0,586.0,1.0,"POLYGON ((-88790.762 -43904.991, -88796.589 -4..."


In [28]:
# Calculate intersected area
geometry_intersect['area_2'] = geometry_intersect.geometry.area

# Calculate the proportion of the tract that intersects each stop
geometry_intersect['area_ratio'] = geometry_intersect['area_2'] / geometry_intersect['area_m2']

In [29]:
# Define demographic and socioeconomic columns to be adjusted by area ratio
cols_to_weight = [
    'total_pop', 'median_household_income', 'employed_pop', 'households_no_vehicle', 
    'total_youth', 
    'inc_extremelylow', 'inc_verylow', 'inc_low', 'lowincome_total', 'jobs_tot_work', 'jobs_tot_home', 'poi_total'
]

# Apply area_ratio
for col in cols_to_weight:
    geometry_intersect[f'{col}_adj'] = geometry_intersect[col] * geometry_intersect['area_ratio']

In [30]:
geometry_intersect.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 115428 entries, 0 to 115427
Data columns (total 57 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   feed_key                     115428 non-null  object  
 1   stop_id                      115428 non-null  object  
 2   stop_name                    115428 non-null  object  
 3   gtfs_dataset_name            115428 non-null  object  
 4   n_arrivals                   115428 non-null  int64   
 5   n_routes                     115428 non-null  int64   
 6   route_id_list                115428 non-null  object  
 7   stop_code                    110895 non-null  object  
 8   pt_geom                      115428 non-null  object  
 9   n_routes_bus                 115428 non-null  int64   
 10  n_routes_rail                115428 non-null  int64   
 11  n_routes_ferry               115428 non-null  int64   
 12  n_routes_other               115428 

In [31]:
# stop_acs_rollup = geometry_intersect.groupby(
#     ['feed_key', 'stop_id'], 
#     as_index=False
# )[[f'{col}_adj' for col in cols_to_weight]].sum()


stop_acs_rollup = geometry_intersect.groupby(
    ['feed_key', 'stop_id', 'gtfs_dataset_name'], 
    as_index=False
)[[f'{col}_adj' for col in cols_to_weight] + ['area_2']].sum()

In [32]:
stop_route_df = agg_with_route_weekday.merge(
    stop_acs_rollup,
    on=['feed_key', 'stop_id', 'gtfs_dataset_name'],
    how='left'
)

In [33]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 24466 entries, 0 to 24465
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   feed_key                     24466 non-null  object  
 1   stop_id                      24466 non-null  object  
 2   stop_name                    24466 non-null  object  
 3   gtfs_dataset_name            24466 non-null  object  
 4   n_arrivals                   24466 non-null  int64   
 5   n_routes                     24466 non-null  int64   
 6   route_id_list                24466 non-null  object  
 7   stop_code                    22953 non-null  object  
 8   pt_geom                      24466 non-null  object  
 9   n_routes_bus                 24466 non-null  int64   
 10  n_routes_rail                24466 non-null  int64   
 11  n_routes_ferry               24466 non-null  int64   
 12  n_routes_other               24466 non-null  int64  

In [34]:
stop_route_df = gpd.GeoDataFrame(
    stop_route_df, 
    geometry='geometry', 
    crs=geometry_intersect.crs
)

In [35]:
# Convert n_arrivals and n_routes to integer
stop_route_df['n_arrivals'] = stop_route_df['n_arrivals'].fillna(0).astype(int)
stop_route_df['n_routes'] = stop_route_df['n_routes'].fillna(0).astype(int)

In [36]:
# # Statewide benchmarks (log1p of density per km², 1st/99th percentile)
# # Computed once from blocks_ca_acs — full CA block group universe, no stops/buffers involved
# BENCHMARKS = {
#     'total_pop':     {'min': 1.0332936988727197, 'max': 9.902553238752084},
#     'jobs_tot_work': {'min': 0.4568467642196446, 'max': 9.54476987861118},
#     'jobs_tot_home': {'min': 0.758723691807346,  'max': 9.133801912505334},
#     'poi_total':     {'min': 0.0,                'max': 5.7166258211721805},
# }

# # Map _adj columns to their benchmark key
# adj_to_key = {
#     "total_pop_adj":     "total_pop",
#     "jobs_tot_work_adj": "jobs_tot_work",
#     "jobs_tot_home_adj": "jobs_tot_home",
#     "poi_total_adj":     "poi_total",
# }

# stop_route_df["area_km2_intersect"] = stop_route_df["area_2"] / 1e6

# for c, key in adj_to_key.items():
#     # density used only for scoring against the statewide benchmark —
#     # stop_route_df[c] (raw _adj count) itself is left untouched
#     density_for_scoring = stop_route_df[c] / stop_route_df["area_km2_intersect"]
#     log_density = np.log1p(density_for_scoring)
#     lo, hi = BENCHMARKS[key]["min"], BENCHMARKS[key]["max"]
#     stop_route_df[c + "_norm"] = ((log_density - lo) / (hi - lo)).clip(0, 1)

# stop_route_df["land_use_index"] = (
#     stop_route_df["total_pop_adj_norm"] +
#     stop_route_df["jobs_tot_work_adj_norm"] +
#     stop_route_df["jobs_tot_home_adj_norm"] +
#     stop_route_df["poi_total_adj_norm"]
# ) / 4

In [37]:
# # Min-max normalize selected columns to a 0–1 scale
# # Each value is rescaled relative to the column minimum and maximum:
# # (x - min) / (max - min)
# # Resulting "_norm" columns allow comparison across variables with different units/scales
# cols = ["total_pop_adj", "jobs_tot_work_adj", "jobs_tot_home_adj", "poi_total_adj"]

# for c in cols:
#     stop_route_df[c + "_norm"] = (
#         (stop_route_df[c] - stop_route_df[c].min()) /
#         (stop_route_df[c].max() - stop_route_df[c].min())
#     )

In [38]:
cols = [
    "total_pop_adj",
    "jobs_tot_work_adj",
    "jobs_tot_home_adj",
    "poi_total_adj"
]

for c in cols:
    stop_route_df[c + "_scaled"] = (
        np.log1p(stop_route_df[c]) /
        (1 + np.log1p(stop_route_df[c]))
    )

stop_route_df["land_use_index"] = (
    stop_route_df["total_pop_adj_scaled"] +
    stop_route_df["jobs_tot_work_adj_scaled"] +
    stop_route_df["jobs_tot_home_adj_scaled"] +
    stop_route_df["poi_total_adj_scaled"]
) / len(cols)

In [39]:
stop_route_df.head(2)

,feed_key,stop_id,stop_name,gtfs_dataset_name,n_arrivals,n_routes,route_id_list,stop_code,pt_geom,n_routes_bus,n_routes_rail,n_routes_ferry,n_routes_other,geometry,total_pop_adj,median_household_income_adj,employed_pop_adj,households_no_vehicle_adj,total_youth_adj,inc_extremelylow_adj,inc_verylow_adj,inc_low_adj,lowincome_total_adj,jobs_tot_work_adj,jobs_tot_home_adj,poi_total_adj,area_2,total_pop_adj_scaled,jobs_tot_work_adj_scaled,jobs_tot_home_adj_scaled,poi_total_adj_scaled,land_use_index
0,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,POINT (-89193.486 -43865.326),2353.845814,64124.523766,810.715007,56.844013,1039.612797,209.287499,149.361716,104.122060,358.649215,108.105704,860.928668,1.335497,513639.385401,0.885900,0.824325,0.871120,0.458940,0.760071
1,019262d9a8ecd3ff0b48b60c5d727ea7,1002,9th St & Nunes Rd,StanRTA Schedule,15,1,[290],1002,POINT(-120.905698 37.551516),1,0,0,0,POINT (-79902.958 -51294.023),877.354189,-4852.094234,350.475928,11.047717,322.183287,53.521146,81.624157,29.800946,135.145303,259.776914,257.978350,2.704287,513639.385401,0.871433,0.847646,0.847485,0.567004,0.783392


In [40]:
stop_route_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 24466 entries, 0 to 24465
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   feed_key                     24466 non-null  object  
 1   stop_id                      24466 non-null  object  
 2   stop_name                    24466 non-null  object  
 3   gtfs_dataset_name            24466 non-null  object  
 4   n_arrivals                   24466 non-null  int64   
 5   n_routes                     24466 non-null  int64   
 6   route_id_list                24466 non-null  object  
 7   stop_code                    22953 non-null  object  
 8   pt_geom                      24466 non-null  object  
 9   n_routes_bus                 24466 non-null  int64   
 10  n_routes_rail                24466 non-null  int64   
 11  n_routes_ferry               24466 non-null  int64   
 12  n_routes_other               24466 non-null  int64  

In [41]:
df_new = stop_route_df.copy()

# log transforms
df_new["log_arrivals"] = np.log(df_new["n_arrivals"].replace(0, np.nan))

In [42]:
# features = [
#     "log_arrivals",
#     "land_use_index",
#     # "households_no_vehicle_adj",
#     "total_youth_adj",
#     "inc_total_lowincome_adj"
# ]

# X = df_new[features].apply(pd.to_numeric, errors="coerce")

In [43]:
# # coefficients from percentile-normalized land_use_index OLS model
# b0 = -5.3337
# b_log_arrivals = 1.1865
# b_land_use = 2.5535
# # b_households_no_vehicle = -3.947e-05
# b_youth = 0.0005
# b_lowincome = 0.0005

# # manual linear prediction
# y_hat = (
#     b0
#     + b_log_arrivals * X["log_arrivals"]
#     + b_land_use * X["land_use_index"]
#     # + b_households_no_vehicle * X["households_no_vehicle_adj"]
#     + b_youth * X["total_youth_adj"]
#     + b_lowincome * X["inc_total_lowincome_adj"]
# )

# # store predictions
# df_new["pred_log_boardings"] = y_hat
# df_new["pred_boardings"] = np.exp(y_hat)

In [44]:
X = df_new[
    ['log_arrivals',
     'land_use_index',
     'total_youth_adj',
     'lowincome_total_adj',
     'n_routes_rail',
     # ferry/other dropped - no variance in training data
    ]
]

# coefficients from mode-count OLS model (rail added, ferry/other dropped)
b0 = -5.3561
b_log_arrivals = 1.1873
b_land_use = 3.6568
b_youth = 0.0007
b_lowincome = 0.0003
b_n_routes_rail = 1.1142

y_hat = (
    b0
    + b_log_arrivals * X["log_arrivals"]
    + b_land_use * X["land_use_index"]
    + b_youth * X["total_youth_adj"]
    + b_lowincome * X["lowincome_total_adj"]
    + b_n_routes_rail * X["n_routes_rail"]
)

df_new["pred_log_boardings"] = y_hat
df_new["pred_boardings"] = np.exp(y_hat)

In [45]:
df_new.head(2)

,feed_key,stop_id,stop_name,gtfs_dataset_name,n_arrivals,n_routes,route_id_list,stop_code,pt_geom,n_routes_bus,n_routes_rail,n_routes_ferry,n_routes_other,geometry,total_pop_adj,median_household_income_adj,employed_pop_adj,households_no_vehicle_adj,total_youth_adj,inc_extremelylow_adj,inc_verylow_adj,inc_low_adj,lowincome_total_adj,jobs_tot_work_adj,jobs_tot_home_adj,poi_total_adj,area_2,total_pop_adj_scaled,jobs_tot_work_adj_scaled,jobs_tot_home_adj_scaled,poi_total_adj_scaled,land_use_index,log_arrivals,pred_log_boardings,pred_boardings
0,019262d9a8ecd3ff0b48b60c5d727ea7,10,Sutter Ave & Nian Way,StanRTA Schedule,56,1,[21],10,POINT(-121.011897 37.617493),1,0,0,0,POINT (-89193.486 -43865.326),2353.845814,64124.523766,810.715007,56.844013,1039.612797,209.287499,149.361716,104.122060,358.649215,108.105704,860.928668,1.335497,513639.385401,0.885900,0.824325,0.871120,0.458940,0.760071,4.025352,3.037952,20.862479
1,019262d9a8ecd3ff0b48b60c5d727ea7,1002,9th St & Nunes Rd,StanRTA Schedule,15,1,[290],1002,POINT(-120.905698 37.551516),1,0,0,0,POINT (-79902.958 -51294.023),877.354189,-4852.094234,350.475928,11.047717,322.183287,53.521146,81.624157,29.800946,135.145303,259.776914,257.978350,2.704287,513639.385401,0.871433,0.847646,0.847485,0.567004,0.783392,2.708050,0.989948,2.691095


In [46]:
cols_to_keep = [
    'stop_id', 'stop_name', 'n_arrivals', 'n_routes', 'route_id_list', 
    'pt_geom', 'gtfs_dataset_name', 'geometry', 'households_no_vehicle_adj',
    'total_youth_adj', 'lowincome_total_adj', 'land_use_index',
    'log_arrivals', 'pred_log_boardings', 'pred_boardings'
]

df_new = df_new[cols_to_keep]
# df_new.to_csv('stops_output_lake_county.csv', index=False)

In [47]:
pred_by_dataset = (
    df_new.groupby("gtfs_dataset_name", as_index=False)["pred_boardings"]
    .sum()
    .rename(columns={"pred_boardings": "total_pred_boardings"})
)

pred_by_dataset

,gtfs_dataset_name,total_pred_boardings
0,Antelope Valley Transit Authority Schedule,4365.548350
1,B-Line Schedule,1692.172483
2,Bay Area 511 Marin Schedule,4085.967374
3,Bay Area 511 Santa Clara Transit Schedule,64077.063127
4,Bay Area 511 Vine Transit Schedule,1266.897999
5,Eastern Sierra Schedule,610.847172
6,El Dorado Schedule,372.830257
7,Humboldt Schedule,828.203736
8,Imperial Valley Transit Schedule,476.740629
9,Kern Schedule,223.940509


- Antelope Valley Transit Authority Schedule : 4,369 average weekday upt
- B-Line Schedule: 2,541 average weekday upt
- Bay Area 511 Marin Schedule: 9764 daily 
- Bay Area 511 Santa Clara Transit Schedule: 57,315 average weekday upt
- Bay Area 511 Vine Transit Schedule	: 1468 average weekday upt
- Eastern Sierra Schedule: 35293 on May 2024 ~ 1138.48 average daily ridership 
- El Dorado Schedule: 170,988 annual upt ~ 468 average daily
- Humboldt Schedule: 367,640 system wide ridership
- Imperial Valley Transit Schedule: 3,461 average weekday upt
- LA Metro Bus Schedule: 661,116 average daily ridership in may 2026
- LA Metro Rail Schedule: 190,600 average daily ridership in may 2026 [link](https://www.metro.net/safety-support/by-the-numbers/)
- Lassen Schedule: ~236.63 average daily upt
- Monterey Salinas Schedule: 7941 average daily boardings in the month of May.
- Morongo Basin Schedule: 510 average daily ridership in 2024
- Mountain Transit Schedule: 1002 average daily ridership in May 2024
- North County Schedule	: 21,835 average systemwide daily ridership in January 2026
- Santa Maria Schedule:  1,970 average weekday upt
- StanRTA sChedule:  8686 average dail;y ridership in May 2026 and 10,594 average weekday upt according to NTD report
- TCRTA :  2,249 average weekday upt
- Victor Valley Schedule:  5,891 average weekday upt
- YARTS: 67,409 annual upt, ~ 184 average

In [50]:
import re
import pandas as pd

# Output Excel file
excel_path = (
    "gs://calitp-analytics-data/data-analyses/ahsc_grant/"
    "ahsc_riderships/AHSC_2026/agency_level_data.xlsx"
)

def clean_sheet_name(name):
    # Excel sheet names: max 31 chars, no []:*?/\
    name = re.sub(r"[\[\]\:\*\?\/\\]", "_", str(name))
    return name[:31]

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for agency_name, group in df_new.groupby("gtfs_dataset_name"):
        sheet_name = clean_sheet_name(agency_name)
        group.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved: {excel_path}")

Saved: gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026/agency_level_data.xlsx


In [55]:
total_pred_boardings*260

734236.2286210338

-  63452 system wide ridership for Lake Transit Authority according to the report.
-  367,640 system wide ridership for Humboldt Transit Authority according to  [December 2024 report](https://hta.org/wp-content/uploads/2026/03/2025.12-Dec-Board-Report.pdf)
-  9764 daily ridership for Marin County Transit District 
 

In [53]:
gdf = df_new.copy()

# Convert to lat/lon
gdf = gdf.to_crs(epsg=4326)

gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

In [54]:
gdf.explore(
    column="pred_boardings",
    cmap="Blues",
    scheme="quantiles",
    k=5,
    marker_kwds={
        "radius": 7,
        "fillOpacity": 0.85,
        "color": "black",
        "weight": 1
    },
    tooltip=[
        "stop_id",
        "stop_name",
        "pred_boardings"
    ],
    tiles="OpenStreetMap",
)

/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/mapclassify/classifiers.py:1767: UserWarning: Not enough unique values in array to form 5 classes. Setting k to 1.
  self.bins = quantile(y, k=k)
/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Cannot apply_along_axis when any iteration dimensions are 0

In [ ]:
# m = gdf.explore(
#     column="pred_boardings",
#     cmap="Blues",
#     scheme="quantiles",
#     k=5,
#     marker_kwds={
#         "radius": 7,
#         "fillOpacity": 0.85,
#         "color": "black",
#         "weight": 1
#     },
#     tooltip=[
#         "stop_id",
#         "stop_name",
#         "pred_boardings"
#     ],
#     tiles="OpenStreetMap",
# )

# m.save("transit_boardings_map.html")